In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import os

class WebScraper:
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

    def fetch_content(self, url):
        """Fetches the HTML content of the given URL."""
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            return None

    def scrape(self, html_content):
        """Parses HTML and extracts usual web parameters."""
        soup = BeautifulSoup(html_content, 'html.parser')
        data = []

        # Extracting common elements: Links, Headings, and Text
        # For a general scraper, we will collect all links and their associated text
        links = soup.find_all('a', href=True)
        
        for link in links:
            row = {
                'Page_Title': soup.title.string if soup.title else 'No Title',
                'Link_Text': link.get_text(strip=True) or 'No Text',
                'URL': link['href'],
                'Tag': link.name
            }
            data.append(row)
        
        return data

    def save_to_csv(self, data, filename="scraped_data.csv"):
        """Saves the list of dictionaries to a CSV file."""
        if not data:
            print("No data found to save.")
            return
        
        df = pd.DataFrame(data)
        df.to_csv(filename, index=False)
        print(f"Successfully saved {len(df)} rows to {filename}")
        return df

In [ ]:
import pandas as pd

# Execution block
def main():
    target_url = input("Please enter the web URL to scrape: ")
    
    # Basic URL validation
    if not target_url.startswith(('http://', 'https://')):
        print("Invalid URL format. Please include http:// or https://")
        return

    scraper = WebScraper()
    print(f"Scraping content from: {target_url}...")
    
    content = scraper.fetch_content(target_url)
    if content:
        results = scraper.scrape(content)
        df = scraper.save_to_csv(results)
        if df is not None:
            display(df.head())

if __name__ == "__main__":
    main()